# ***Pytorch-Code-Mixed-Precision-Training-And-Training-on-Multi-Core-TPU***

## **Fine Tuned Bert Ensemble Model (CoreBert + Discharge Bert) For MultiClass Classification.**

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/877787/1000 instances Mortality-Prediction/1000 instances Mortality-Prediction Val.csv
/kaggle/input/877787/1000 instances Mortality-Prediction/1000 instances Mortality-Prediction Train.csv
/kaggle/input/877787/1000 instances Mortality-Prediction/1000 instances Mortality-Prediction Test.csv
/kaggle/input/877787/Opus Mortality Prediction 1000 Instances/Opus Mortality Prediction 1000 Instances/OPUS 1000 instances test MP
/kaggle/input/877787/Opus Mortality Prediction 1000 Instances/Opus Mortality Prediction 1000 Instances/OPUS 1000 instances Val MP
/kaggle/input/877787/Opus Mortality Prediction 1000 Instances/Opus Mortality Prediction 1000 Instances/OPUS 1000 instances Train MP


In [2]:
#pip install flash-attn

In [3]:
#pip install git+https://github.com/HazyResearch/flash-attention.git


In [4]:
import torch
from torch.utils.data import DataLoader
from transformers import BertForSequenceClassification, BertTokenizer, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, AutoConfig
from torch.utils.data import DataLoader
from torch import nn
from transformers import AdamW, get_linear_schedule_with_warmup
from tqdm import tqdm
import os
from torch.nn import functional as F
import torch.quantization
import torch.nn.utils.prune as prune


# Loading Dataset

In [5]:
train=pd.read_csv("/kaggle/input/877787/Opus Mortality Prediction 1000 Instances/Opus Mortality Prediction 1000 Instances/OPUS 1000 instances Train MP")
test=pd.read_csv("/kaggle/input/877787/Opus Mortality Prediction 1000 Instances/Opus Mortality Prediction 1000 Instances/OPUS 1000 instances test MP")
val=pd.read_csv("/kaggle/input/877787/Opus Mortality Prediction 1000 Instances/Opus Mortality Prediction 1000 Instances/OPUS 1000 instances Val MP")

In [6]:
train

,Unnamed: 0,machine
0,0,প্রধান অভিযোগ: দেখা বর্তমান অসুস্থতার মোট ক্ষত...
1,1,"প্রধান অভিযোগ: বাম দিকের দুর্বলতা, বর্তমান অসু..."
2,2,প্রধান অভিযোগ: রোগীটি কোরোনারি ধূমপানের রোগে উ...
3,3,প্রধান অভিযোগ: শ্বাসপ্রশ্বাসের অসুখ বর্তমান অস...
4,4,প্রধান অভিযোগ: বর্তমান অসুস্থতা: রোগীর বয়স ৫৩...
...,...,...
796,796,প্রধান অভিযোগ: বর্তমান অসুস্থতা: রোগীর বয়স ৪৫...
797,797,প্রধান অভিযোগ: মানসিক অবস্থা এবং হাইপোটেনশন বর...
798,798,প্রধান অভিযোগ: [**Country 3399*] থেকে [*OTCURE...
799,799,প্রধান অভিযোগ: বর্তমান অসুস্থতা: [*2151-5-25*]...


In [7]:
test

,Unnamed: 0,machine
0,0,প্রধান অভিযোগ: বর্তমান অসুস্থতা: চিকিৎসা ইতিহা...
1,1,প্রধান অভিযোগ: নিউমোনিয়ায় বর্তমান অসুস্থতা: ...
2,2,প্রধান অভিযোগ: বর্তমান রোগের প্রতারণা: সংক্ষেপ...
3,3,"প্রধান অভিযোগ: আব্ডোমিনাল ব্যথা, জ্বর বর্তমান ..."
4,4,প্রধান অভিযোগ: বুকের ব্যথা বর্তমান অসুস্থতা: [...
...,...,...
96,96,প্রধান অভিযোগ: বর্তমান অসুস্থতা: ৫৪ মিটার cad ...
97,97,প্রধান অভিযোগ: ট্রাচেস্টোমি ট্রাচেল স্টিনোসিস ...
98,98,প্রধান অভিযোগ: [**cc contact info 82463*] বর্ত...
99,99,প্রধান অভিযোগ: sepic sective sective pression:...


In [8]:
val

,Unnamed: 0,machine
0,0,"প্রধান অভিযোগ: সিঙ্কোপ, কাঁচা, বমি, ডায়ের বর্..."
1,1,প্রধান অভিযোগ: আব্ডোমিনাল ব্যথা বর্তমান জরুরী ...
2,2,প্রধান অভিযোগ: বর্তমান অসুস্থতা: [***এটি পরিচি...
3,3,প্রধান অভিযোগ: Bradycardia বর্তমান অসুস্থতা: M...
4,4,প্রধান অভিযোগ: আরো বৃদ্ধি পাচ্ছে তাচিপ্নিয়ার ...
...,...,...
95,95,প্রধান অভিযোগ: mitrale and tricuspid valve reg...
96,96,প্রধান অভিযোগ: হিপ ভেঙে যাওয়া বর্তমান অসুস্থত...
97,97,প্রধান অভিযোগ: প্যানক্রিটাইটিস বর্তমান অসুস্থত...
98,98,প্রধান অভিযোগ: এস/পি পতন বর্তমান অসুস্থতা: htn...


In [9]:
train_label=pd.read_csv('/kaggle/input/877787/1000 instances Mortality-Prediction/1000 instances Mortality-Prediction Train.csv')
val_label=pd.read_csv('/kaggle/input/877787/1000 instances Mortality-Prediction/1000 instances Mortality-Prediction Val.csv')
test_label=pd.read_csv('/kaggle/input/877787/1000 instances Mortality-Prediction/1000 instances Mortality-Prediction Test.csv')

In [10]:
train_label

,Unnamed: 0,text,los_label
0,0,CHIEF COMPLAINT: total vision loss\n\nPRESENT ...,0
1,1,"CHIEF COMPLAINT: left sided weakness, acute on...",0
2,2,CHIEF COMPLAINT: The patient presents with cor...,0
3,3,CHIEF COMPLAINT: Respiratory failure\n\nPRESEN...,0
4,4,CHIEF COMPLAINT: \n\nPRESENT ILLNESS: The pati...,0
...,...,...,...
796,796,CHIEF COMPLAINT: \n\nPRESENT ILLNESS: The pati...,1
797,797,CHIEF COMPLAINT: Altered Mental Status and hyp...,1
798,798,CHIEF COMPLAINT: right lower extremity swellin...,0
799,799,CHIEF COMPLAINT: \n\nPRESENT ILLNESS: The pati...,0


In [11]:
test_label

,Unnamed: 0,text,los_label
0,0,CHIEF COMPLAINT: \n\nPRESENT ILLNESS: \n\nMEDI...,0
1,1,CHIEF COMPLAINT: Pneumonia\n\nPRESENT ILLNESS:...,0
2,2,CHIEF COMPLAINT: Delusions\n\nPRESENT ILLNESS:...,0
3,3,"CHIEF COMPLAINT: abdominal pain, fever\n\nPRES...",0
4,4,CHIEF COMPLAINT: Chest pain\n\nPRESENT ILLNESS...,0
...,...,...,...
96,96,CHIEF COMPLAINT: Epistaxis\n\nPRESENT ILLNESS:...,0
97,97,CHIEF COMPLAINT: Post tracheostomy tracheal st...,0
98,98,CHIEF COMPLAINT: CC:[**CC Contact Info 82463**...,0
99,99,CHIEF COMPLAINT: Septic shock\n\nPRESENT ILLNE...,0


In [12]:
val_label

,Unnamed: 0,text,los_label
0,0,"CHIEF COMPLAINT: syncope, chills, nausea, vomi...",1
1,1,CHIEF COMPLAINT: Abdominal pain Hypertensive e...,0
2,2,CHIEF COMPLAINT: \n\nPRESENT ILLNESS: Mr. [**K...,0
3,3,CHIEF COMPLAINT: Bradycardia\n\nPRESENT ILLNES...,0
4,4,CHIEF COMPLAINT: Admit to MICU for increased t...,0
...,...,...,...
95,95,CHIEF COMPLAINT: Mitral and tricuspid valve re...,0
96,96,CHIEF COMPLAINT: Hip fracture\n\nPRESENT ILLNE...,0
97,97,CHIEF COMPLAINT: pancreatitis\n\nPRESENT ILLNE...,0
98,98,CHIEF COMPLAINT: s/p fall\n\nPRESENT ILLNESS: ...,0


In [13]:
train['los_label']=train_label.los_label
val['los_label']=val_label.los_label
test['los_label']=test_label.los_label

In [14]:
train

,Unnamed: 0,machine,los_label
0,0,প্রধান অভিযোগ: দেখা বর্তমান অসুস্থতার মোট ক্ষত...,0
1,1,"প্রধান অভিযোগ: বাম দিকের দুর্বলতা, বর্তমান অসু...",0
2,2,প্রধান অভিযোগ: রোগীটি কোরোনারি ধূমপানের রোগে উ...,0
3,3,প্রধান অভিযোগ: শ্বাসপ্রশ্বাসের অসুখ বর্তমান অস...,0
4,4,প্রধান অভিযোগ: বর্তমান অসুস্থতা: রোগীর বয়স ৫৩...,0
...,...,...,...
796,796,প্রধান অভিযোগ: বর্তমান অসুস্থতা: রোগীর বয়স ৪৫...,1
797,797,প্রধান অভিযোগ: মানসিক অবস্থা এবং হাইপোটেনশন বর...,1
798,798,প্রধান অভিযোগ: [**Country 3399*] থেকে [*OTCURE...,0
799,799,প্রধান অভিযোগ: বর্তমান অসুস্থতা: [*2151-5-25*]...,0


In [15]:
test

,Unnamed: 0,machine,los_label
0,0,প্রধান অভিযোগ: বর্তমান অসুস্থতা: চিকিৎসা ইতিহা...,0
1,1,প্রধান অভিযোগ: নিউমোনিয়ায় বর্তমান অসুস্থতা: ...,0
2,2,প্রধান অভিযোগ: বর্তমান রোগের প্রতারণা: সংক্ষেপ...,0
3,3,"প্রধান অভিযোগ: আব্ডোমিনাল ব্যথা, জ্বর বর্তমান ...",0
4,4,প্রধান অভিযোগ: বুকের ব্যথা বর্তমান অসুস্থতা: [...,0
...,...,...,...
96,96,প্রধান অভিযোগ: বর্তমান অসুস্থতা: ৫৪ মিটার cad ...,0
97,97,প্রধান অভিযোগ: ট্রাচেস্টোমি ট্রাচেল স্টিনোসিস ...,0
98,98,প্রধান অভিযোগ: [**cc contact info 82463*] বর্ত...,0
99,99,প্রধান অভিযোগ: sepic sective sective pression:...,0


In [16]:
val

,Unnamed: 0,machine,los_label
0,0,"প্রধান অভিযোগ: সিঙ্কোপ, কাঁচা, বমি, ডায়ের বর্...",1
1,1,প্রধান অভিযোগ: আব্ডোমিনাল ব্যথা বর্তমান জরুরী ...,0
2,2,প্রধান অভিযোগ: বর্তমান অসুস্থতা: [***এটি পরিচি...,0
3,3,প্রধান অভিযোগ: Bradycardia বর্তমান অসুস্থতা: M...,0
4,4,প্রধান অভিযোগ: আরো বৃদ্ধি পাচ্ছে তাচিপ্নিয়ার ...,0
...,...,...,...
95,95,প্রধান অভিযোগ: mitrale and tricuspid valve reg...,0
96,96,প্রধান অভিযোগ: হিপ ভেঙে যাওয়া বর্তমান অসুস্থত...,0
97,97,প্রধান অভিযোগ: প্যানক্রিটাইটিস বর্তমান অসুস্থত...,0
98,98,প্রধান অভিযোগ: এস/পি পতন বর্তমান অসুস্থতা: htn...,0


In [17]:
train.rename(columns={"machine": "text"},inplace=True)
val.rename(columns={"machine": "text"},inplace=True)
test.rename(columns={"machine": "text"},inplace=True)

In [18]:
train

,Unnamed: 0,text,los_label
0,0,প্রধান অভিযোগ: দেখা বর্তমান অসুস্থতার মোট ক্ষত...,0
1,1,"প্রধান অভিযোগ: বাম দিকের দুর্বলতা, বর্তমান অসু...",0
2,2,প্রধান অভিযোগ: রোগীটি কোরোনারি ধূমপানের রোগে উ...,0
3,3,প্রধান অভিযোগ: শ্বাসপ্রশ্বাসের অসুখ বর্তমান অস...,0
4,4,প্রধান অভিযোগ: বর্তমান অসুস্থতা: রোগীর বয়স ৫৩...,0
...,...,...,...
796,796,প্রধান অভিযোগ: বর্তমান অসুস্থতা: রোগীর বয়স ৪৫...,1
797,797,প্রধান অভিযোগ: মানসিক অবস্থা এবং হাইপোটেনশন বর...,1
798,798,প্রধান অভিযোগ: [**Country 3399*] থেকে [*OTCURE...,0
799,799,প্রধান অভিযোগ: বর্তমান অসুস্থতা: [*2151-5-25*]...,0


In [19]:
train.los_label.unique()

array([0, 1])

# Maximum Length of a note

max(len(x) for x in train['text'])

# Avg Length of a Note

import statistics
statistics.mean([len(x) for x in train['text']])

### Model HyperParameters

In [20]:
batch_size = 16
max_tokens = 512
epochs = 200
best_roc_auc = 0.0
min_delta = 0.0001
early_stopping_count = 0
early_stopping_patience = 3
gradient_accumulation_steps = 2
lr = 3e-5
weight_decay = 0.01
pruning_ratio = 0.3
dropout_prob = 0.2

# Make Ensemble

class EnsembleModel(nn.Module):
    def __init__(self, model1, model2):
        super(EnsembleModel, self).__init__()
        self.model1 = model1
        self.model2 = model2

    def forward(self, input_ids, attention_mask):
        output1 = self.model1(input_ids, attention_mask=attention_mask)[0]
        output2 = self.model2(input_ids, attention_mask=attention_mask)[0]
        avg_output = (output1 + output2) / 2.00
        return avg_output

## Load Model From Hugging Face

config = AutoConfig.from_pretrained('bvanaken/CORe-clinical-outcome-biobert-v1',
                                    num_labels=4,
                                    hidden_dropout_prob=dropout_prob,
                                    attention_probs_dropout_prob=dropout_prob)
core_model = AutoModelForSequenceClassification.from_pretrained('bvanaken/CORe-clinical-outcome-biobert-v1' ,config=config)

In [21]:
pip install git+https://github.com/csebuetnlp/normalizer

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


  Cloning https://github.com/csebuetnlp/normalizer to /tmp/pip-req-build-jl1t7kx8
  Running command git clone --filter=blob:none --quiet https://github.com/csebuetnlp/normalizer /tmp/pip-req-build-jl1t7kx8
  Resolved https://github.com/csebuetnlp/normalizer to commit d405944dde5ceeacb7c2fd3245ae2a9dea5f35c9
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for normalizer: filename=normalizer-0.0.1-py3-none-any.whl size=6859 sha256=916be86752f4eeb83ab08bfc7cf01d771a8e53319b83794fc54908a1fc34f27e
  Stored in directory: /tmp/pip-ephem-wheel-cache-ylxtzdbi/wheels/2e/79/9c/cd96d490298305d51d2da11484bb2c25fd1f759a6906708282
  Created wheel for emoji: filename=emoji-1.4.2-py3-none-any.whl size=186459 sha256=929f28485a515b298295b5ceca9019b8b895949d

In [22]:
from normalizer import normalize

# BanglaBert Medium

In [23]:
model_name="csebuetnlp/banglabert"
model_name

'csebuetnlp/banglabert'

In [24]:
config = AutoConfig.from_pretrained(model_name,
                                    num_labels=2, #hidden_dropout_prob=dropout_prob,
                                    )             #attention_probs_dropout_prob=dropout_prob

# Load the pre-trained model with the specified configuration
model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
print(f"hidden_dropout_prob: {config.hidden_dropout_prob}")
print(f"attention_probs_dropout_prob: {config.attention_probs_dropout_prob}")

hidden_dropout_prob: 0.1
attention_probs_dropout_prob: 0.1


## Push Model To device ("Tpu")

import torch_xla
import torch_xla.core.xla_model as xm

# Check if TPU is available
device = xm.xla_device()
print(device)
model = model.to(device)

> **Use Cuda For Mixed Precision Training** 

## Dependiencies for TPU

import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
import torch_xla.distributed.xla_multiprocessing as xmp
import torch_xla.utils.utils as xu
import torch_xla.core.xla_model as xm
import torch_xla.core.functions as xf

#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")    ^^^For MIXED PRECISION 
import torch_xla
import torch_xla.core.xla_model as xm

# Check if TPU is available
device = xm.xla_device()
print(device)

ensemble_model = EnsembleModel(core_model, discharge_model)
ensemble_model = ensemble_model.to(device)

# Push Model To device 

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Intializing Tokenizer

In [27]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

# Dataset Class

In [28]:
class LosDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

### Normalization

In [29]:
train['text']=train['text'].apply(normalize)
val['text']=val['text'].apply(normalize)
test['text']=test['text'].apply(normalize)

### Tokenization

In [30]:
train_encodings = tokenizer(train['text'].tolist(),truncation=True, padding=True, max_length = max_tokens)                       #Expects list[str] so train['text'].tolist()
val_encodings = tokenizer(val['text'].tolist(), truncation=True, padding=True,  max_length = max_tokens)
test_encodings = tokenizer(test['text'].tolist(), truncation=True, padding=True , max_length = max_tokens)

In [31]:
train_dataset = LosDataset(train_encodings, train['los_label'].tolist())
val_dataset = LosDataset(val_encodings, val['los_label'].tolist())
test_dataset = LosDataset(test_encodings, test['los_label'].tolist())

len(train_dataset.labels[:])

## Loading Model to Dataloader 

In [32]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,pin_memory=True)

## Setting Up Optimizer And Scheduler

In [33]:
warmup_ratio = 0.1
total_training_steps = (epochs * len(train_loader)) // gradient_accumulation_steps
num_warmup_steps = int(warmup_ratio * total_training_steps)

In [34]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps= total_training_steps
)

## Loading Trained Model's state dict  (IF NEEDED)

### FOR MIXED PRECISION TRANING

In [41]:
state_dict = torch.load('/kaggle/working/Bangla Bert_epoch_0_roc_0.5267.pth')                 
# Load the state dict into your model
model.load_state_dict(state_dict)
model = model.to(device) 

print('Loaded Trained Model state dict  ')

/tmp/ipykernel_30/3202450090.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('/kaggle/working/Bangla Bert_epoch_0_roc_0.5267.pth')


Loaded Trained Model state dict  


### FOR MULTI CORE TPU

state_dict = torch.load('/kaggle/input/uuuu/pytorch/default/1/CoreDischarge Model_epoch_4roc_0.7123073312793169.pth',map_location=torch.device('cpu')) # Pushing to Cpu
device = xm.xla_device()                  

# Load the state dict into your model
ensemble_model.load_state_dict(state_dict)
ensemble_model = ensemble_model.to(device)   # Pushing to TPU

### Single Train Loader

In [35]:
for step,batch in enumerate(train_loader):
    print(step,end="\n \n")
    print(batch)
    
    if step==0:
        break

0
 
{'input_ids': tensor([[    2,  1415,  1660,  ...,    72,  3853,     3],
        [    2,  1415,  1660,  ...,     0,     0,     0],
        [    2,  1415,  1660,  ...,     0,     0,     0],
        ...,
        [    2,  1415,  1660,  ..., 12544,   452,     3],
        [    2,  1415,  1660,  ...,   205,   886,     3],
        [    2,  1415,  1660,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0])}


# Train and Eval Loop


In [36]:
ensemble_name='Bangla Bert'
from tqdm import tqdm

### Model Parameters

In [37]:
model.parameters

<bound method Module.parameters of ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias

### See If Model loaded on Tpu

param_device = next(model.parameters()).device
print(f"Model is loaded on device: {param_device}")

# Cpu Code

best_roc_auc = 0.0
min_delta = 0.0001
for epoch in range(epochs):
    model.train()
    train_loss = 0
    para_loader = train_loader  # Removed MpDeviceLoader
    for step, batch in enumerate(tqdm(para_loader)):
        if step % gradient_accumulation_steps == 0:
            optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Extract logits from SequenceClassifierOutput
        outputs = model(input_ids, attention_mask)
        logits = outputs.logits  # Extract logits for loss calculation #########################
        
        loss = nn.CrossEntropyLoss()(logits, labels)                   #################################
        (loss / gradient_accumulation_steps).backward()
        train_loss += loss.item()
        if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
            optimizer.step()  # Replaced xm.optimizer_step with optimizer.step
            scheduler.step()

    model.eval()
    val_loss = 0
    val_preds = []
    val_labels = []
    para_val_loader = val_loader  # Removed MpDeviceLoader
    with torch.no_grad():
        for batch in tqdm(para_val_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Extract logits from SequenceClassifierOutput
            outputs = model(input_ids, attention_mask)
            logits = outputs.logits  # Extract logits for loss calculation ###################
            
            loss = nn.CrossEntropyLoss()(logits, labels)     ##########################
            val_loss += loss.item()
            val_preds.append(F.softmax(logits, dim=1).cpu().numpy())
            val_labels.append(labels.cpu().numpy())

    val_preds = np.concatenate(val_preds)
    val_labels = np.concatenate(val_labels)
    val_loss /= len(val_loader)
    train_loss /= len(train_loader)
    print(f'Epoch: {epoch+1}/{epochs}, Training Loss: {train_loss}, Validation Loss: {val_loss}')

    # Calculate metrics
    val_preds_class = np.argmax(val_preds, axis=1)
    accuracy = accuracy_score(val_labels, val_preds_class)
    recall = recall_score(val_labels, val_preds_class, average='weighted')
    precision = precision_score(val_labels, val_preds_class, average='weighted')
    f1 = f1_score(val_labels, val_preds_class, average='weighted')
    micro_f1 = f1_score(val_labels, val_preds_class, average='micro')
    macro_roc_auc = roc_auc_score(val_labels, val_preds, multi_class='ovo', average='macro')

    print(f'Accuracy: {accuracy}, Recall: {recall}, Precision: {precision}, F1: {f1}, Micro F1: {micro_f1}, Macro Roc Auc: {macro_roc_auc} , Best Roc Auc: {best_roc_auc}')

    # Implement early stopping

    if macro_roc_auc - best_roc_auc < min_delta:
        early_stopping_count += 1
        print(f'EarlyStopping counter: {early_stopping_count} out of {early_stopping_patience}')
        if early_stopping_count >= early_stopping_patience:
            print('Early stopping')
            break
    else:
        best_roc_auc = macro_roc_auc
        early_stopping_count = 0
        torch.save(model.state_dict(), f"{ensemble_name}_epoch_{epoch}roc_{best_roc_auc}.pth")


# **Using Multi Core  TPU for Training**

for epoch in range(epochs):
    model.train()
    train_loss = 0
    para_loader = pl.MpDeviceLoader(train_loader, device)  #~~~~~~~~~~~~~~~~~~~~~  HERE
    for step, batch in enumerate(tqdm(para_loader)):       #~~~~~~~~~~~~~~~~~~~~~  HERE
        optimizer.zero_grad() if step % gradient_accumulation_steps == 0 else None
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask)
        logits = outputs.logits  # Extract logits for loss calculation #########################
        
        loss = nn.CrossEntropyLoss()(logits, labels)     
        (loss / gradient_accumulation_steps).backward()
        train_loss += loss.item()
        if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
            xm.optimizer_step(optimizer)                   #~~~~~~~~~~~~~~~~~~~~~  HERE
            scheduler.step()

    model.eval()
    val_loss = 0
    val_preds = []
    val_labels = []
    para_val_loader = pl.MpDeviceLoader(val_loader, device)       #~~~~~~~~~~~~~~~~~~~~~  HERE
    with torch.no_grad():
        for batch in tqdm(para_val_loader):                       #~~~~~~~~~~~~~~~~~~~~~  HERE
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask)
            logits = outputs.logits  # Extract logits for loss calculation ###################
            
            loss = nn.CrossEntropyLoss()(logits, labels)  
            val_loss += loss.item()
            val_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
            val_labels.append(labels.cpu().numpy())


    val_preds = np.concatenate(val_preds)
    val_labels = np.concatenate(val_labels)
    val_loss /= len(val_loader)
    train_loss /= len(train_loader)
    print(f'Epoch: {epoch+1}/{epochs}, Training Loss: {train_loss}, Validation Loss: {val_loss}')

    # Calculate metrics
    val_preds_class = np.argmax(val_preds, axis=1)
    accuracy = accuracy_score(val_labels, val_preds_class)
    recall = recall_score(val_labels, val_preds_class, average='weighted')
    precision = precision_score(val_labels, val_preds_class, average='weighted')
    f1 = f1_score(val_labels, val_preds_class, average='weighted')
    micro_f1 = f1_score(val_labels, val_preds_class, average='micro')
    macro_roc_auc = roc_auc_score(val_labels, val_preds, multi_class='ovo', average='macro')

    print(f'Accuracy: {accuracy}, Recall: {recall}, Precision: {precision}, F1: {f1}, Micro F1: {micro_f1}, Macro Roc Auc: {macro_roc_auc}')


    # Implement early stopping
    if macro_roc_auc - best_roc_auc < min_delta:
        early_stopping_count += 1
        print(f'EarlyStopping counter: {early_stopping_count} out of {early_stopping_patience}')
        if early_stopping_count >= early_stopping_patience:
            print('Early stopping')
            break
    else:
        best_roc_auc = macro_roc_auc
        early_stopping_count = 0
        xm.save(ensemble_model.state_dict(), f"{ensemble_name}_epoch_{epoch}roc_{best_roc_auc}.pth")

# **Using Mixed Precision Technique for Training**

### *Use Cuda for this*

## MAIN

In [39]:
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score, f1_score
import numpy as np

# Constants
best_roc_auc = 0.0
min_delta = 0.0001
early_stopping_count = 0
early_stopping_patience = 3  # Set patience value

# Training loop
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for step, batch in enumerate(tqdm(train_loader)):
        if step % gradient_accumulation_steps == 0:
            optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Mixed precision training
        with torch.amp.autocast('cuda'):
            outputs = model(input_ids, attention_mask)
            logits = outputs.logits  # Extract logits
            loss = nn.CrossEntropyLoss()(logits, labels)
            loss = loss / gradient_accumulation_steps  # Normalize loss for accumulation

        # Backpropagation
        loss.backward()
        train_loss += loss.item()

        if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
            # Clip gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # Optimizer step and scheduler step
            optimizer.step()
            scheduler.step()

    # Average training loss
    train_loss /= len(train_loader)

    # Evaluation loop (NO MIXED PRECISION)
    model.eval()
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            # Standard precision evaluation
            outputs = model(input_ids, attention_mask)
            logits = outputs.logits  # Extract logits
            loss = nn.CrossEntropyLoss()(logits, labels)  # No loss division in validation

            val_loss += loss.item()
            probabilities = F.softmax(logits, dim=1)  # Convert logits to probabilities
            val_preds.append(probabilities.cpu().numpy())
            val_labels.append(labels.cpu().numpy())

    # Calculate validation metrics
    val_preds = np.concatenate(val_preds)
    val_labels = np.concatenate(val_labels)
    val_loss /= len(val_loader)

    # Debugging check to ensure probabilities sum to 1
    if not np.allclose(val_preds.sum(axis=1), 1.0, atol=1e-6):
        print("Warning: val_preds do not sum to 1. Fixing with normalization.")
        val_preds = val_preds / val_preds.sum(axis=1, keepdims=True)

    val_preds_class = np.argmax(val_preds, axis=1)
    accuracy = accuracy_score(val_labels, val_preds_class)
    recall = recall_score(val_labels, val_preds_class)
    precision = precision_score(val_labels, val_preds_class)
    f1 = f1_score(val_labels, val_preds_class)
    roc_auc = roc_auc_score(val_labels, val_preds[:, 1])

    print(f"Epoch: {epoch+1}/{epochs}, Training Loss: {train_loss}, Validation Loss: {val_loss}")
    print(
        f"Accuracy: {accuracy}, Recall: {recall}, Precision: {precision}, F1: {f1}, Micro F1: {f1}, "
        f"Roc Auc: {roc_auc}, Best Roc Auc: {best_roc_auc}"
    )

    # Early stopping logic
    if roc_auc - best_roc_auc < min_delta:
        early_stopping_count += 1
        print(f"EarlyStopping counter: {early_stopping_count} out of {early_stopping_patience}")
        if early_stopping_count >= early_stopping_patience:
            print("Early stopping")
            break
    else:
        best_roc_auc = roc_auc
        early_stopping_count = 0
        torch.save(model.state_dict(), f"{ensemble_name}_epoch_{epoch}_roc_{best_roc_auc:.4f}.pth")


100%|██████████| 7/7 [00:03<00:00,  1.99it/s]
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch: 1/200, Training Loss: 0.2758003683651195, Validation Loss: 0.4596167802810669
Accuracy: 0.9, Recall: 0.0, Precision: 0.0, F1: 0.0, Micro F1: 0.0, Macro Roc Auc: 0.5266666666666666, Best Roc Auc: 0.0


100%|██████████| 7/7 [00:03<00:00,  1.84it/s]
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch: 2/200, Training Loss: 0.20697522630878523, Validation Loss: 0.33427869209221434
Accuracy: 0.9, Recall: 0.0, Precision: 0.0, F1: 0.0, Micro F1: 0.0, Macro Roc Auc: 0.51, Best Roc Auc: 0.5266666666666666
EarlyStopping counter: 1 out of 3


100%|██████████| 7/7 [00:04<00:00,  1.72it/s]
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch: 3/200, Training Loss: 0.1898527706370634, Validation Loss: 0.30514161927359446
Accuracy: 0.9, Recall: 0.0, Precision: 0.0, F1: 0.0, Micro F1: 0.0, Macro Roc Auc: 0.49777777777777776, Best Roc Auc: 0.5266666666666666
EarlyStopping counter: 2 out of 3


100%|██████████| 7/7 [00:03<00:00,  1.83it/s]

Epoch: 4/200, Training Loss: 0.16622614393047258, Validation Loss: 0.3020673042961529
Accuracy: 0.9, Recall: 0.0, Precision: 0.0, F1: 0.0, Micro F1: 0.0, Macro Roc Auc: 0.4533333333333333, Best Roc Auc: 0.5266666666666666
EarlyStopping counter: 3 out of 3
Early stopping



/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## Learning rate Optimization 


import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoConfig, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score, f1_score
import numpy as np
from tqdm import tqdm

# Constants
batch_size = 16
max_tokens = 512
epochs = 200
min_delta = 0.0001
early_stopping_patience = 3
gradient_accumulation_steps = 2
weight_decay = 0.01
pruning_ratio = 0.3
dropout_prob = 0.2
model_name = "csebuetnlp/banglabert"
warmup_ratio = 0.1
SEED = 42

# Learning rates to test
learning_rates = [3e-5, 4e-5, 5e-5]
best_lr = None
best_overall_roc_auc = 0.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

for lr in learning_rates:
    print(f"Testing learning rate: {lr}")
    config = AutoConfig.from_pretrained(
        model_name,
        num_labels=4,
        # hidden_dropout_prob=dropout_prob,
        # attention_probs_dropout_prob=dropout_prob,
    )

    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Calculate warmup steps and total training steps
    total_training_steps = (epochs * len(train_loader)) // gradient_accumulation_steps
    num_warmup_steps = int(warmup_ratio * total_training_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=total_training_steps,
    )

    best_roc_auc = 0.0
    early_stopping_count = 0

    # Training loop
    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for step, batch in enumerate(tqdm(train_loader)):
            if step % gradient_accumulation_steps == 0:
                optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            # Mixed precision training
            with torch.amp.autocast('cuda'):
                outputs = model(input_ids, attention_mask)
                logits = outputs.logits
                loss = nn.CrossEntropyLoss()(logits, labels)
                loss = loss / gradient_accumulation_steps

            loss.backward()
            train_loss += loss.item()

            if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()

        train_loss /= len(train_loader)

        # Evaluation loop
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []

        with torch.no_grad():
            for batch in tqdm(val_loader):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs = model(input_ids, attention_mask)
                logits = outputs.logits
                loss = nn.CrossEntropyLoss()(logits, labels)

                val_loss += loss.item()
                probabilities = F.softmax(logits, dim=1)
                val_preds.append(probabilities.cpu().numpy())
                val_labels.append(labels.cpu().numpy())

        val_preds = np.concatenate(val_preds)
        val_labels = np.concatenate(val_labels)
        val_loss /= len(val_loader)

        if not np.allclose(val_preds.sum(axis=1), 1.0, atol=1e-6):
            print("Warning: val_preds do not sum to 1. Fixing with normalization.")
            val_preds = val_preds / val_preds.sum(axis=1, keepdims=True)

        val_preds_class = np.argmax(val_preds, axis=1)
        accuracy = accuracy_score(val_labels, val_preds_class)
        recall = recall_score(val_labels, val_preds_class, average="weighted")
        precision = precision_score(val_labels, val_preds_class, average="weighted")
        f1 = f1_score(val_labels, val_preds_class, average="weighted")
        micro_f1 = f1_score(val_labels, val_preds_class, average="micro")
        macro_roc_auc = roc_auc_score(val_labels, val_preds, multi_class="ovo", average="macro")

        print(f"Epoch: {epoch + 1}/{epochs}, Training Loss: {train_loss}, Validation Loss: {val_loss}")
        print(
            f"Accuracy: {accuracy}, Recall: {recall}, Precision: {precision}, F1: {f1}, Micro F1: {micro_f1}, "
            f"Macro Roc Auc: {macro_roc_auc}, Best Roc Auc: {best_roc_auc}"
        )

        if macro_roc_auc - best_roc_auc < min_delta:
            early_stopping_count += 1
            print(f"EarlyStopping counter: {early_stopping_count} out of {early_stopping_patience}")
            if early_stopping_count >= early_stopping_patience:
                print("Early stopping")
                break
        else:
            best_roc_auc = macro_roc_auc
            early_stopping_count = 0
            torch.save(model.state_dict(), f"{ensemble_name}_lr_{lr}_epoch_{epoch}_roc_{best_roc_auc:.4f}.pth")

    print(f"Finished training with learning rate {lr}, Best ROC AUC: {best_roc_auc}")

    if best_roc_auc > best_overall_roc_auc:
        best_overall_roc_auc = best_roc_auc
        best_lr = lr

print(f"Best learning rate: {best_lr}, Best ROC AUC: {best_overall_roc_auc}")


model.eval()
val_loss = 0
val_preds = []
val_labels = []

with torch.no_grad():
    for batch in tqdm(val_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass without mixed precision
        outputs = model(input_ids, attention_mask)
        logits = outputs.logits  # Extract logits
        loss = nn.CrossEntropyLoss()(logits, labels)

        val_loss += loss.item()
        
        # Convert logits to probabilities and normalize
        probabilities = F.softmax(logits, dim=1)
        val_preds.append(probabilities.cpu().numpy())
        val_labels.append(labels.cpu().numpy())

# Calculate validation metrics
val_preds = np.concatenate(val_preds)
val_labels = np.concatenate(val_labels)
val_loss /= len(val_loader)

# Debugging check to ensure probabilities sum to 1
if not np.allclose(val_preds.sum(axis=1), 1.0, atol=1e-6):
    print("Warning: val_preds do not sum to 1. Fixing with normalization.")
    val_preds = val_preds / val_preds.sum(axis=1, keepdims=True)

val_preds_class = np.argmax(val_preds, axis=1)
accuracy = accuracy_score(val_labels, val_preds_class)
recall = recall_score(val_labels, val_preds_class, average="weighted")
precision = precision_score(val_labels, val_preds_class, average="weighted")
f1 = f1_score(val_labels, val_preds_class, average="weighted")
micro_f1 = f1_score(val_labels, val_preds_class, average="micro")
macro_roc_auc = roc_auc_score(val_labels, val_preds, multi_class="ovo", average="macro")

print(f"Epoch: {epoch+1}/{epochs}, Training Loss: {train_loss}, Validation Loss: {val_loss}")
print(
    f"Accuracy: {accuracy}, Recall: {recall}, Precision: {precision}, F1: {f1}, Micro F1: {micro_f1}, "
    f"Macro Roc Auc: {macro_roc_auc}, Best Roc Auc: {best_roc_auc}"
)

# Early stopping logic
if macro_roc_auc - best_roc_auc < min_delta:
    early_stopping_count += 1
    print(f"EarlyStopping counter: {early_stopping_count} out of {early_stopping_patience}")
    if early_stopping_count >= early_stopping_patience:
        print("Early stopping")
        break
else:
    best_roc_auc = macro_roc_auc
    early_stopping_count = 0
    torch.save(model.state_dict(), f"{ensemble_name}_epoch_{epoch}_roc_{best_roc_auc:.4f}.pth")


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler

# Constants
best_roc_auc = 0.0
min_delta = 0.0001
early_stopping_count = 0
early_stopping_patience = 3  # Set patience value
scaler = torch.amp.GradScaler('cuda')  # Mixed precision gradient scaler

# Training loop
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for step, batch in enumerate(tqdm(train_loader)):
        if step % gradient_accumulation_steps == 0:
            optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Mixed precision training
        with torch.amp.autocast('cuda'):
            outputs = model(input_ids, attention_mask)
            logits = outputs.logits  # Extract logits
            loss = nn.CrossEntropyLoss()(logits, labels)
            loss = loss / gradient_accumulation_steps  # Normalize loss for accumulation

        # Backpropagation with scaling
        scaler.scale(loss).backward()
        train_loss += loss.item()

        if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
            # Unscale gradients and clip
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # Optimizer step and scale update
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

    # Average training loss
    train_loss /= len(train_loader)

    # Evaluation loop
    model.eval()
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            # Mixed precision evaluation
            with torch.amp.autocast('cuda'):
                outputs = model(input_ids, attention_mask)
                logits = outputs.logits  # Extract logits
                loss = nn.CrossEntropyLoss()(logits, labels)
            
            val_loss += loss.item()
            val_preds.append(F.softmax(logits, dim=1).cpu().numpy())
            val_labels.append(labels.cpu().numpy())

    # Calculate validation metrics
    val_preds = np.concatenate(val_preds)
    val_labels = np.concatenate(val_labels)
    val_loss /= len(val_loader)

    val_preds_class = np.argmax(val_preds, axis=1)
    accuracy = accuracy_score(val_labels, val_preds_class)
    recall = recall_score(val_labels, val_preds_class, average="weighted")
    precision = precision_score(val_labels, val_preds_class, average="weighted")
    f1 = f1_score(val_labels, val_preds_class, average="weighted")
    micro_f1 = f1_score(val_labels, val_preds_class, average="micro")
    macro_roc_auc = roc_auc_score(val_labels, val_preds, multi_class="ovo", average="macro")

    print(f"Epoch: {epoch+1}/{epochs}, Training Loss: {train_loss}, Validation Loss: {val_loss}")
    print(
        f"Accuracy: {accuracy}, Recall: {recall}, Precision: {precision}, F1: {f1}, Micro F1: {micro_f1}, "
        f"Macro Roc Auc: {macro_roc_auc}, Best Roc Auc: {best_roc_auc}"
    )

    # Early stopping logic
    if macro_roc_auc - best_roc_auc < min_delta:
        early_stopping_count += 1
        print(f"EarlyStopping counter: {early_stopping_count} out of {early_stopping_patience}")
        if early_stopping_count >= early_stopping_patience:
            print("Early stopping")
            break
    else:
        best_roc_auc = macro_roc_auc
        early_stopping_count = 0
        torch.save(model.state_dict(), f"{ensemble_name}_epoch_{epoch}_roc_{best_roc_auc:.4f}.pth")


from torch.cuda.amp import autocast, GradScaler # IMPORTS


scaler = torch.amp.GradScaler("cuda")          #~~~~~~~~~~~~~~~~~~~~~  HERE

for epoch in range(epochs):
    
    ensemble_model.train()
    train_loss = 0
    for step, batch in enumerate(tqdm(train_loader)):
        optimizer.zero_grad() if step % gradient_accumulation_steps == 0 else None
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Use autocast for mixed precision
        with torch.amp.autocast("cuda"):        #~~~~~~~~~~~~~~~~~~~~~  HERE
            outputs = ensemble_model(input_ids, attention_mask)
            loss = nn.CrossEntropyLoss()(outputs, labels)
            loss = loss / gradient_accumulation_steps

        # Scale the loss and call backward()
        scaler.scale(loss).backward()          #~~~~~~~~~~~~~~~~~~~~~  HERE
        train_loss += loss.item()

        if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
            # Unscales the gradients of optimizer's assigned params in-place
            scaler.unscale_(optimizer)
            
            # Clip gradients
            torch.nn.utils.clip_grad_norm_(ensemble_model.parameters(), max_norm=1.0)
            
            # Optimizer step 
            scaler.step(optimizer)            #~~~~~~~~~~~~~~~~~~~~~  HERE
            
            # Update the scale for next iteration
            scaler.update()
            scheduler.step()
            

    ensemble_model.eval()
    val_loss = 0
    val_preds = []
    val_labels = []
    with torch.no_grad():
        for batch in tqdm(val_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            #with torch.amp.autocast("cuda"):
            outputs = ensemble_model(input_ids, attention_mask)
            loss = nn.CrossEntropyLoss()(outputs, labels)
            
            val_loss += loss.item()
            val_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
            val_labels.append(labels.cpu().numpy())

    val_preds = np.concatenate(val_preds)
    val_labels = np.concatenate(val_labels)
    val_loss /= len(val_loader)
    train_loss /= len(train_loader)
    print(f'Epoch: {epoch+1}/{epochs}, Training Loss: {train_loss}, Validation Loss: {val_loss}') #,

    # Calculate metrics
    val_preds_class = np.argmax(val_preds, axis=1)
    accuracy = accuracy_score(val_labels, val_preds_class)
    recall = recall_score(val_labels, val_preds_class, average='weighted')
    precision = precision_score(val_labels, val_preds_class, average='weighted')
    f1 = f1_score(val_labels, val_preds_class, average='weighted')
    micro_f1 = f1_score(val_labels, val_preds_class, average='micro')
    macro_roc_auc = roc_auc_score(val_labels, val_preds, multi_class='ovo', average='macro')
    print(f'Accuracy: {accuracy}, Recall: {recall}, Precision: {precision}, F1: {f1}, Micro F1: {micro_f1}, Macro Roc Auc: {macro_roc_auc}')

    # Implement early stopping
    if macro_roc_auc - best_roc_auc < min_delta:
        early_stopping_count += 1
        print(f'EarlyStopping counter: {early_stopping_count} out of {early_stopping_patience}')
        if early_stopping_count >= early_stopping_patience:
            print('Early stopping')
            break
    else:
        best_roc_auc = macro_roc_auc
        early_stopping_count = 0
        torch.save(ensemble_model.state_dict(), f"{ensemble_name}_epoch_{epoch}roc_{best_roc_auc}.pth")

### Model evaluation on Test Dataset

In [42]:
# Test Loop
model.eval()

test_loss = 0
test_preds = []
test_labels = []

# Disable gradient computations for testing
with torch.no_grad():
    for batch in tqdm(test_loader):  # Assuming test_loader is already defined
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass with mixed precision
        
        outputs = model(input_ids, attention_mask)
        logits = outputs.logits  # Extract logits

        # Calculate loss
        loss = nn.CrossEntropyLoss()(logits, labels)
        test_loss += loss.item()

        # Collect predictions and labels
        test_preds.append(F.softmax(logits, dim=1).cpu().numpy())
        test_labels.append(labels.cpu().numpy())

# Concatenate predictions and labels
test_preds = np.concatenate(test_preds)
test_labels = np.concatenate(test_labels)

# Calculate average test loss
test_loss /= len(test_loader)

# Calculate metrics
test_preds_class = np.argmax(test_preds, axis=1)
accuracy = accuracy_score(test_labels, test_preds_class)
recall = recall_score(test_labels, test_preds_class)
precision = precision_score(test_labels, test_preds_class)
f1 = f1_score(test_labels, test_preds_class)
roc_auc = roc_auc_score(test_labels, test_preds[:, 1])

# Print metrics
print(f"Test Loss: {test_loss}")
print(f"Accuracy: {accuracy}, Recall: {recall}, Precision: {precision}, F1: {f1}")
print(f"Micro F1: {f1}, Macro Roc AUC: {roc_auc}")


100%|██████████| 7/7 [00:03<00:00,  1.94it/s]

Test Loss: 0.4696788489818573
Accuracy: 0.8910891089108911, Recall: 0.0, Precision: 0.0, F1: 0.0
Micro F1: 0.0, Macro Roc AUC: 0.23535353535353531



/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


# Test Loop
model.eval()

test_loss = 0
test_preds = []
test_labels = []

# Prepare data loader for parallelized device loading
para_test_loader = pl.MpDeviceLoader(test_loader, device)

# Disable gradient computations for testing
with torch.no_grad():
    for batch in tqdm(para_test_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids, attention_mask)
        logits = outputs.logits  # Extract logits

        # Calculate loss
        loss = nn.CrossEntropyLoss()(logits, labels)
        test_loss += loss.item()

        # Collect predictions and labels
        test_preds.append(F.softmax(logits, dim=1).cpu().numpy())
        test_labels.append(labels.cpu().numpy())

# Concatenate predictions and labels
test_preds = np.concatenate(test_preds)
test_labels = np.concatenate(test_labels)

# Calculate average test loss
test_loss /= len(test_loader)

# Calculate metrics
test_preds_class = np.argmax(test_preds, axis=1)
accuracy = accuracy_score(test_labels, test_preds_class)
recall = recall_score(test_labels, test_preds_class, average='weighted')
precision = precision_score(test_labels, test_preds_class, average='weighted')
f1 = f1_score(test_labels, test_preds_class, average='weighted')
micro_f1 = f1_score(test_labels, test_preds_class, average='micro')
macro_roc_auc = roc_auc_score(test_labels, test_preds, multi_class='ovo', average='macro')

# Print metrics
print(f"Test Loss: {test_loss}")
print(f"Accuracy: {accuracy}, Recall: {recall}, Precision: {precision}, F1: {f1}")
print(f"Micro F1: {micro_f1}, Macro Roc AUC: {macro_roc_auc}")


#ensemble_model.to('cpu')
#device = 'cpu'

# Put the model in evaluation mode
model.eval()

test_preds = []
test_labels = []

# Iterate over test data
para_test_loader = pl.MpDeviceLoader(test_loader, device)
with torch.no_grad():
    for batch in tqdm(para_test_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask)
        test_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
        test_labels.append(labels.cpu().numpy())
test_preds = np.concatenate(test_preds)
test_labels = np.concatenate(test_labels)

# Calculate metrics
test_preds_class = np.argmax(test_preds, axis=1)
accuracy = accuracy_score(test_labels, test_preds_class)
recall = recall_score(test_labels, test_preds_class, average='weighted')
precision = precision_score(test_labels, test_preds_class, average='weighted')
f1 = f1_score(test_labels, test_preds_class, average='weighted')
micro_f1 = f1_score(test_labels, test_preds_class, average='micro')
macro_roc_auc = roc_auc_score(test_labels, test_preds, multi_class='ovo', average='macro')

print(f'Accuracy: {accuracy}, Recall: {recall}, Precision: {precision}, F1: {f1}, Micro F1: {micro_f1}, Macro Roc Auc: {macro_roc_auc}')